# 13 — Vector Search RAG Pipeline

**Project:** GroundedNutriRec  
**Scope:** Build a controlled RAG (Retrieval-Augmented Generation) pipeline where the LLM explains food recommendations **only** using retrieved evidence from the Food Knowledge Base.  

## Objectives
1. Load the food knowledge base created in Notebook 12.
2. Generate dense embeddings using **SentenceTransformers** (`all-MiniLM-L6-v2`).
3. Build a **FAISS** vector index for fast approximate nearest-neighbour search.
4. Implement a **retrieval** function that, given a user query, returns the top-k most relevant food documents.
5. Construct a **grounded prompt** that feeds retrieved evidence to an LLM and instructs it to answer **only** from the provided context.
6. Demonstrate end-to-end RAG with sample queries.
7. Persist the FAISS index for reuse.

## 1. Install Dependencies

Install the required packages if not already present.

In [1]:
# Install required packages (run once)
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

for pkg in ['sentence-transformers', 'faiss-cpu', 'transformers', 'google-genai']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        print(f'Installing {pkg}...')
        install(pkg)

print('All dependencies ready.')

Installing sentence-transformers...
Installing faiss-cpu...
Installing google-genai...
All dependencies ready.


## 2. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import json
import faiss
import warnings
import pickle
import time
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import pipeline

warnings.filterwarnings('ignore')

# -- Paths --
KB_CSV_PATH    = Path('dataset/archive_3/food_knowledge_base.csv')
FAISS_DIR      = Path('dataset/archive_3/faiss_index')
FAISS_DIR.mkdir(parents=True, exist_ok=True)
INDEX_PATH     = FAISS_DIR / 'food_kb.index'
METADATA_PATH  = FAISS_DIR / 'food_kb_meta.pkl'

# -- Embedding Model --
EMBED_MODEL_NAME = 'all-MiniLM-L6-v2'   # 384-dim, fast & effective

# -- HuggingFace Generation Model --
print('Loading local LLM (TinyLlama)...')
llm_pipeline = pipeline('text-generation', model='TinyLlama/TinyLlama-1.1B-Chat-v1.0', max_new_tokens=256, temperature=0.3, do_sample=True)

print(f'Knowledge base  : {KB_CSV_PATH}')
print(f'FAISS index dir : {FAISS_DIR}')
print(f'Embedding model : {EMBED_MODEL_NAME}')
print(f'LLM             : TinyLlama-1.1B (Local)')
print('Setup complete.')

## 3. Load Food Knowledge Base

In [3]:
print('Loading knowledge base...')
kb_df_full = pd.read_csv(KB_CSV_PATH)
print(f'  Full KB shape: {kb_df_full.shape}')

# Drop any rows with missing documents
kb_df_full = kb_df_full.dropna(subset=['document']).reset_index(drop=True)

# --- Sample top-rated recipes for CPU-friendly embedding ---
# Encoding 231K documents on CPU would take hours.
# We select the top 20,000 recipes by average rating (with at least 1 review)
# to build a high-quality, representative vector store.
SAMPLE_SIZE = 20_000

kb_df = (
    kb_df_full[kb_df_full['review_count'] > 0]
    .sort_values('avg_rating', ascending=False)
    .head(SAMPLE_SIZE)
    .reset_index(drop=True)
)

print(f'  Sampled {len(kb_df):,} top-rated recipes (from {len(kb_df_full):,} total)')
print(f'  Rating range: {kb_df["avg_rating"].min():.2f} - {kb_df["avg_rating"].max():.2f}')

documents = kb_df['document'].tolist()
print(f'\nSample document:')
print('=' * 60)
print(documents[0])

Loading knowledge base...
  Full KB shape: (231637, 11)
  Sampled 20,000 top-rated recipes (from 231,637 total)
  Rating range: 5.00 - 5.00

Sample document:
Recipe: arriba   baked winter squash mexican style
Ingredients: winter squash, mexican seasoning, mixed spice, honey, butter, olive oil, salt
Calories: 51.5 kcal | Protein: 2.0 %DV | Fat: 0.0 %DV
Prep Time: 55 minutes
Average Rating: 5.0/5 (3 reviews)
Review Summary:  I used an acorn squash and recipe#137681 Sweet Mexican spice blend. Only used 1 tsp honey & 1 tsp butter between both halves,, sprinkled the squash liberally with the spice mix. Baked covered for 45 minutes uncovered or 15.  I basted the squash   with the the butter/honey from the cavity  allowing it to get a golden color.  Lovely Squash recipe Thanks Cookgirl


## 4. Generate SentenceTransformer Embeddings

We encode every knowledge-base document into a dense vector using `all-MiniLM-L6-v2` (384 dimensions).  
This model provides a good trade-off between speed and semantic quality.

> **Note:** Encoding ~230 k documents may take several minutes on CPU. A GPU will speed this up significantly.

In [ ]:
print(f'Loading SentenceTransformer model: {EMBED_MODEL_NAME} ...')
model = SentenceTransformer(EMBED_MODEL_NAME)
print(f'  Embedding dimension: {model.get_sentence_embedding_dimension()}')

# ── Encode documents ──────────────────────────────────
BATCH_SIZE = 256

print(f'\nEncoding {len(documents):,} documents (batch_size={BATCH_SIZE})...')
start = time.time()

embeddings = model.encode(
    documents,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # L2-normalise for cosine similarity via inner-product
)

elapsed = time.time() - start
print(f'\nEncoding complete in {elapsed:.1f}s')
print(f'Embeddings shape: {embeddings.shape}')

Loading SentenceTransformer model: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 5. Build FAISS Vector Index

We use **FAISS IndexFlatIP** (Inner Product) because our embeddings are L2-normalised, so inner product equals cosine similarity.  
For very large collections, an approximate index (e.g. `IndexIVFFlat`) would be faster, but the flat index guarantees exact results.

In [ ]:
dim = embeddings.shape[1]
print(f'Building FAISS IndexFlatIP (dim={dim})...')

index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))

print(f'  Vectors in index: {index.ntotal:,}')
print('FAISS index built successfully.')

## 6. Persist FAISS Index & Metadata

In [ ]:
# ── Save FAISS index ──
faiss.write_index(index, str(INDEX_PATH))
print(f'FAISS index saved -> {INDEX_PATH}')

# ── Save metadata (recipe_ids + documents) ──
metadata = {
    'recipe_ids': kb_df['recipe_id'].tolist(),
    'documents':  documents,
    'names':      kb_df['name'].tolist(),
}
with open(METADATA_PATH, 'wb') as f:
    pickle.dump(metadata, f)
print(f'Metadata saved   -> {METADATA_PATH}')
print('\n[OK] Index persisted.')

## 7. Semantic Retrieval Function

Given a natural-language query, encode it and retrieve the **top-k** most similar documents from the FAISS index.

In [ ]:
def retrieve(query: str, model, index, metadata, top_k: int = 5):
    """
    Retrieve the top-k most relevant food documents for a given query.
    
    Parameters
    ----------
    query    : str  – natural-language user query
    model    : SentenceTransformer model
    index    : FAISS index
    metadata : dict with 'documents', 'recipe_ids', 'names'
    top_k    : int  – number of results to return
    
    Returns
    -------
    list[dict] – each dict has keys: rank, score, recipe_id, name, document
    """
    # Encode query
    q_emb = model.encode([query], normalize_embeddings=True).astype(np.float32)
    
    # Search
    scores, indices = index.search(q_emb, top_k)
    
    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        results.append({
            'rank':      rank,
            'score':     float(score),
            'recipe_id': metadata['recipe_ids'][idx],
            'name':      metadata['names'][idx],
            'document':  metadata['documents'][idx],
        })
    return results

print('Retrieval function defined. [OK]')

## 8. Test Retrieval with Sample Queries

In [ ]:
test_queries = [
    'high protein low fat chicken recipe',
    'quick breakfast under 15 minutes',
    'healthy vegetarian pasta with low calories',
    'best rated chocolate dessert',
]

for query in test_queries:
    print(f'\n{"═" * 80}')
    print(f'QUERY: {query}')
    print(f'{"═" * 80}')
    results = retrieve(query, model, index, metadata, top_k=3)
    for r in results:
        print(f'\n  [{r["rank"]}] Score: {r["score"]:.4f}  |  {r["name"]}')
        # Show first 200 chars of document
        snippet = r['document'][:200].replace('\n', ' | ')
        print(f'      {snippet}...')

## 9. Grounded RAG Prompt Construction

The key to a **controlled RAG** pipeline is the prompt template.  
We instruct the LLM to:
1. **Only** use the retrieved food evidence.
2. **Cite** which recipe(s) the recommendation is based on.
3. **Refuse** to answer if the evidence does not support the query.

In [ ]:
RAG_SYSTEM_PROMPT = """\
You are a nutrition-aware food recommendation assistant.
You MUST answer the user's question ONLY using the retrieved food evidence provided below.
Do NOT use any external knowledge or make up information.

Rules:
1. Base your answer strictly on the EVIDENCE section.
2. Cite the recipe name(s) you reference.
3. Include relevant nutritional information (calories, protein, fat) from the evidence.
4. If the evidence does not contain enough information to answer, say:
   "I don't have enough evidence to answer this question."
5. Keep your answer concise and helpful.
"""

def build_rag_prompt(query: str, retrieved_docs: list) -> str:
    """
    Build a grounded RAG prompt from the user query and retrieved documents.
    """
    evidence_block = '\n\n'.join(
        f'--- Evidence {doc["rank"]} (score: {doc["score"]:.4f}) ---\n{doc["document"]}'
        for doc in retrieved_docs
    )
    
    # TinyLlama chat format
    prompt = (
        f'<|system|>\n'
        f'{RAG_SYSTEM_PROMPT}\n'
        f'=== EVIDENCE ===\n'
        f'{evidence_block}</s>\n'
        f'<|user|>\n'
        f'{query}</s>\n'
        f'<|assistant|>\n'
    )
    return prompt


def generate_answer(query: str, retrieved_docs: list) -> str:
    """
    Generate a grounded answer using local TinyLlama.
    """
    prompt = build_rag_prompt(query, retrieved_docs)
    output = llm_pipeline(prompt, return_full_text=False)
    return output[0]['generated_text'].strip()

print('RAG prompt builder defined.')
print('Local LLM answer generator defined.')

## 10. End-to-End RAG Demonstration

We demonstrate the full pipeline: **Query -> Retrieve -> Prompt -> Gemini LLM -> Grounded Answer**

In [ ]:
# -- Full pipeline demo --
demo_query = 'I want a high-protein, low-calorie meal that is quick to prepare'

print(f'USER QUERY: {demo_query}')
print(f'{chr(9472) * 80}\n')

# Step 1: Retrieve relevant documents
retrieved = retrieve(demo_query, model, index, metadata, top_k=5)
print(f'Retrieved {len(retrieved)} documents:\n')
for r in retrieved:
    print(f'  [{r["rank"]}] {r["name"]}  (score={r["score"]:.4f})')

# Step 2: Generate grounded answer using Gemini
print(f'\n{"=" * 80}')
print('GEMINI GROUNDED ANSWER:')
print(f'{"=" * 80}\n')

answer = generate_answer(demo_query, retrieved)
print(answer)

In [ ]:
# -- Additional queries demo --
additional_queries = [
    'What is a good low-fat dessert with chocolate?',
    'Suggest a quick vegetarian dinner under 30 minutes',
    'I need a high-protein breakfast recipe',
]

for q in additional_queries:
    print(f'\n{"=" * 80}')
    print(f'QUERY: {q}')
    print(f'{"=" * 80}\n')
    
    results = retrieve(q, model, index, metadata, top_k=5)
    print('Top retrieved recipes:')
    for r in results:
        print(f'  [{r["rank"]}] {r["name"]} (score={r["score"]:.4f})')
    
    print(f'\nGemini Answer:')
    print(f'{"-" * 40}')
    ans = generate_answer(q, results)
    print(ans)

## 11. Utility: Load Persisted Index

For future use, here is how to reload the saved FAISS index and metadata without re-encoding.

In [ ]:
def load_faiss_index(index_path, metadata_path):
    """Load a previously saved FAISS index and its metadata."""
    idx = faiss.read_index(str(index_path))
    with open(metadata_path, 'rb') as f:
        meta = pickle.load(f)
    print(f'Loaded FAISS index with {idx.ntotal:,} vectors.')
    return idx, meta

# Demo reload
reloaded_index, reloaded_meta = load_faiss_index(INDEX_PATH, METADATA_PATH)

# Quick check
test_results = retrieve('simple pasta recipe', model, reloaded_index, reloaded_meta, top_k=3)
for r in test_results:
    print(f'  [{r["rank"]}] {r["name"]} (score={r["score"]:.4f})')

print('\n[OK] Reloaded index works correctly.')

## 12. Summary

| Component | Details |
|-----------|--------|
| Embedding Model | `all-MiniLM-L6-v2` (384-dim, SentenceTransformers) |
| Vector Store | FAISS `IndexFlatIP` (exact cosine similarity via normalised inner product) |
| Knowledge Base | 20,000 top-rated recipes (sampled from ~231K for CPU efficiency) |
| Retrieval | Top-k semantic nearest-neighbour search |
| RAG Prompt | Grounded — LLM must answer only from retrieved evidence |
| Persisted Artifacts | `food_kb.index` (FAISS) + `food_kb_meta.pkl` (metadata) |

### Key Design Decisions
- **Controlled generation**: The system prompt explicitly forbids the LLM from using external knowledge.
- **Citation required**: The LLM must cite recipe names, ensuring traceability.
- **Evidence-gated**: If the retrieved evidence does not support the query, the LLM is instructed to say so.

This pipeline can be extended with:
- **Re-ranking** (e.g. cross-encoder) for higher precision.
- **Hybrid search** (BM25 + dense) for better recall.
- **Streaming LLM output** for better user experience.
- **User feedback loop** to continuously improve retrieval quality.